### Leer Datos JSON de la carpeta "Locations"
1. Consultar archivo único
2. Consultar la lista de archivos utilizando caracteres comodín
3. Consultar todos los archivos de una carpeta
4. Seleccionar metadatos de archivo
5. Crear Tabla en el esquema bronze
6. Crear vista temporal(Temporary View)
7. Crear Vista Temporal Global(Global Temporary View)
8. Crear Vista(View) Permanente en Unity Catalog

In [0]:
%fs ls /Volumes/zenviro/raw/operational_data/locations/

#### 1. Consultar archivo único

In [0]:
location_df = spark.read.json("/Volumes/zenviro/raw/operational_data/locations/locations_2025_10.json")

In [0]:
location_df = spark.read.format("json").load("/Volumes/zenviro/raw/operational_data/locations/locations_2025_10.json")

#### 2. Consultar la lista de archivos utilizando caracteres comodín

In [0]:
location_df = spark.read.format("json").load("/Volumes/zenviro/raw/operational_data/locations/locations_2025_*.json")
display(location_df)

#### 3. Consultar todos los archivos de una carpeta

In [0]:
location_df = spark.read.format("json").load("/Volumes/zenviro/raw/operational_data/locations/")
display(location_df)

#### 4. Seleccionar metadatos de archivo

In [0]:
location_with_matadata_df = location_df.select("_metadata.file_path", "*")
display(location_with_matadata_df)

#### 5. Crear Tabla en el esquema bronze

In [0]:
location_with_matadata_df.write.format("delta").mode("overwrite").saveAsTable("zenviro.bronze.locations_py")

In [0]:
location_with_matadata_df.writeTo("zenviro.bronze.locations_py").createOrReplace()

In [0]:
%sql
SELECT * FROM zenviro.bronze.locations_py;

#### 6. Crear vista temporal(Temporary View)

In [0]:
location_with_matadata_df.createOrReplaceTempView("tv_locations_py")

In [0]:
%sql
SELECT * FROM tv_locations_py;

#### 7. Crear Vista Temporal Global(Global Temporary View)

In [0]:
location_with_matadata_df.createOrReplaceGlobalTempView("gtv_locations_py")

In [0]:
%sql
SELECT * FROM global_temp.gtv_locations_py;

#### 8. Crear Vista(View) Permanente en Unity Catalog

In [0]:
spark.sql("""
          CREATE OR REPLACE VIEW zenviro.bronze.v_locations_py
          AS
          SELECT * FROM zenviro.bronze.locations_py
          """)

In [0]:
%sql
SELECT * FROM zenviro.bronze.v_locations_py